In [5]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [31]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
import joblib

In [2]:
df = pd.read_csv("dataset.csv")
df.head

<bound method NDFrame.head of                                       Disease             Symptom_1  \
0                            Fungal infection               itching   
1                            Fungal infection             skin_rash   
2                            Fungal infection               itching   
3                            Fungal infection               itching   
4                            Fungal infection               itching   
...                                       ...                   ...   
4915  (vertigo) Paroymsal  Positional Vertigo              vomiting   
4916                                     Acne             skin_rash   
4917                  Urinary tract infection   burning_micturition   
4918                                Psoriasis             skin_rash   
4919                                 Impetigo             skin_rash   

                  Symptom_2              Symptom_3                  Symptom_4  \
0                 skin_rash   nodal_

In [3]:
df.fillna("NoSymptoms", inplace=True)

In [7]:
symp_columns = [col for col in df.columns if col.startswith("Symptom")]
df["allsymps"] = df[symp_columns].values.tolist()
df[["allsymps"]].head()

,allsymps
0,"[itching, skin_rash, nodal_skin_eruptions, ..."
1,"[ skin_rash, nodal_skin_eruptions, dischromi..."
2,"[itching, nodal_skin_eruptions, dischromic _..."
3,"[itching, skin_rash, dischromic _patches, No..."
4,"[itching, skin_rash, nodal_skin_eruptions, N..."


In [13]:
def combine_symptoms(row):
    return [symptom for symptom in row if symptom != "NoSymptoms"]

df["allsymps"] = df[symp_columns].apply(combine_symptoms, axis=1)

df[["allsymps"]].head()

,allsymps
0,"[itching, skin_rash, nodal_skin_eruptions, ..."
1,"[ skin_rash, nodal_skin_eruptions, dischromi..."
2,"[itching, nodal_skin_eruptions, dischromic _..."
3,"[itching, skin_rash, dischromic _patches]"
4,"[itching, skin_rash, nodal_skin_eruptions]"


In [18]:
mlb = MultiLabelBinarizer()
x = mlb.fit_transform(df["allsymps"])

In [19]:
print("No_Symptom" in mlb.classes_)

False


In [20]:
x_df = pd.DataFrame(x, columns = mlb.classes_)
x_df.head()

,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin,itching
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [21]:
le = LabelEncoder()
y = le.fit_transform(df["Disease"])
print(y.shape)
print(y[7:])

(4920,)
[15 15 15 ... 38 35 27]


In [25]:
for i in range(5):
    print(df["Disease"][i], "→", y[i])

Fungal infection → 15
Fungal infection → 15
Fungal infection → 15
Fungal infection → 15
Fungal infection → 15


In [28]:
x_train, x_test, y_train, y_test = (
train_test_split(x,y,test_size=0.2,random_state=42,stratify=y) 
)

In [30]:
print("X_train shape:", x_train.shape)
print("X_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (3936, 131)
X_test shape: (984, 131)
y_train shape: (3936,)
y_test shape: (984,)


In [36]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(x_train,y_train)
lr_y_pred = lr.predict(x_test)
print("Accuracy for Logistic:", accuracy_score(y_test,lr_y_pred))
print (classification_report(y_test,lr_y_pred))

Accuracy for Logistic: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        24
           2       1.00      1.00      1.00        24
           3       1.00      1.00      1.00        24
           4       1.00      1.00      1.00        24
           5       1.00      1.00      1.00        24
           6       1.00      1.00      1.00        24
           7       1.00      1.00      1.00        24
           8       1.00      1.00      1.00        24
           9       1.00      1.00      1.00        24
          10       1.00      1.00      1.00        24
          11       1.00      1.00      1.00        24
          12       1.00      1.00      1.00        24
          13       1.00      1.00      1.00        24
          14       1.00      1.00      1.00        24
          15       1.00      1.00      1.00        24
          16       1.00      1.00      1.00        24


In [38]:
from sklearn.naive_bayes import BernoulliNB

nb = BernoulliNB()
nb.fit(x_train,y_train)
nb_y_pred = nb.predict(x_test)

print("Accuracy For NB:", accuracy_score(y_test,nb_y_pred))
print(classification_report(y_test,nb_y_pred))

Accuracy For NB: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        24
           2       1.00      1.00      1.00        24
           3       1.00      1.00      1.00        24
           4       1.00      1.00      1.00        24
           5       1.00      1.00      1.00        24
           6       1.00      1.00      1.00        24
           7       1.00      1.00      1.00        24
           8       1.00      1.00      1.00        24
           9       1.00      1.00      1.00        24
          10       1.00      1.00      1.00        24
          11       1.00      1.00      1.00        24
          12       1.00      1.00      1.00        24
          13       1.00      1.00      1.00        24
          14       1.00      1.00      1.00        24
          15       1.00      1.00      1.00        24
          16       1.00      1.00      1.00        24
      

In [33]:
rf.fit(x_train,y_train)


RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [32]:
rf = RandomForestClassifier(
    n_estimators=200,  
    random_state=42,
    n_jobs=-1             
)

In [34]:
y_pred = rf.predict(x_test)
print("Accuracy:", accuracy_score(y_test,y_pred))

Accuracy: 1.0


In [35]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        24
           2       1.00      1.00      1.00        24
           3       1.00      1.00      1.00        24
           4       1.00      1.00      1.00        24
           5       1.00      1.00      1.00        24
           6       1.00      1.00      1.00        24
           7       1.00      1.00      1.00        24
           8       1.00      1.00      1.00        24
           9       1.00      1.00      1.00        24
          10       1.00      1.00      1.00        24
          11       1.00      1.00      1.00        24
          12       1.00      1.00      1.00        24
          13       1.00      1.00      1.00        24
          14       1.00      1.00      1.00        24
          15       1.00      1.00      1.00        24
          16       1.00      1.00      1.00        24
          17       1.00    